In [ ]:
from embedder import SentenceTransformerEmbeddings
from langchain_astradb import AstraDBVectorStore
from google.cloud import firestore
import uuid
import os
import dotenv

dotenv.load_dotenv(override=True)

embedder = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2",
    device=None,
    
)
    
    # Initialize vector store
vector_store = AstraDBVectorStore(
    collection_name="puranas",
    embedding=embedder,
    token=os.getenv("ASTRA_DB_APPLICATION_TOKEN"),
    api_endpoint=os.getenv("ASTRA_DB_API_ENDPOINT"),
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
query = "Who is Bhanu Vinayak ?"

results = vector_store.similarity_search_with_score(
    query,
    k=4,
)

# Display results
print(f"\n{'='*60}")
print("RETRIEVED DOCUMENTS")
print(f"{'='*60}")

# Since results is a list of tuples (Document, score)
for i, (doc, score) in enumerate(results):
    # Convert distance to similarity score (assuming score is distance)
    similarity_score = 1 - score if score <= 1.0 else score
    
    # Extract metadata
    metadata = doc.metadata if hasattr(doc, 'metadata') else {}
    doc_id = getattr(doc, 'id', f"doc_{i+1}")
    
    print(f"\nResult {i+1}:")
    print(f"ID: {doc_id}")
    print(f"Book: {metadata.get('book', 'Unknown')}")
    print(f"Chapter: {metadata.get('chapter_info', 'Unknown')}")
    print(f"Similarity Score: {similarity_score:.4f}")
    print(f"Distance: {score:.4f}" if score <= 1.0 else f"Score: {score:.4f}")
    print(f"Content preview: {doc.page_content[:150]}...")


RETRIEVED DOCUMENTS

Result 1:
ID: da0e9db0-e68b-4bfd-aa22-5ccfcc42d63a
Book: Mudgal_Puran_Khand_6
Chapter: Chapter 43 : The Story of Bhanu Vinayaka
Similarity Score: 0.1921
Distance: 0.8079
Content preview: Chapter 43 : The Story of Bhanu Vinayaka

Salutations to Shri Ganesha.

The Primordial Power (Adi Shakti) said: Kashyapa had a wife named Vinata, who ...

Result 2:
ID: c6a7cb15-152f-46c8-aed9-e48dcd810504
Book: Mudgal_Puran_Khand_6
Chapter: Chapter 43 : The Story of Bhanu Vinayaka
Similarity Score: 0.1950
Distance: 0.8050
Content preview: After worshipping Ganesha, he went to the Sun, bowed to him, and narrated the entire story. Thus, Aruna established the supreme Ganapati there, worshi...

Result 3:
ID: b877a65f-fa8f-406c-adc7-1721b69f4079
Book: Ganesh_Puran_Upasana_Khand
Chapter: Chapter 27 : The Description of Rukmangada’s Coronation
Similarity Score: 0.2044
Distance: 0.7956
Content preview: Then, Vinayaka appeared, took the King by the hand, and said, "O King, you are liberat

In [3]:
import os
import dotenv
from augmentation import Augmentation

dotenv.load_dotenv(override=True)


aug = Augmentation(
    vector_store=vector_store, 
)


result = aug.augment(
    query="Who is Bhanu Vinayak ?",
    top_k=4
)

print(result["answer"])
print("\nModel used:", result["model"])


Generating response using OpenRouter...

Trying model: nvidia/nemotron-3.5-lightning
Based on the supplied scriptural passages, particularly from the Mudgala Purana, Bhanu Vinayaka is presented as a distinct form and incarnation of Ganesha whose narrative and theological significance are articulated in Chapter 43 of the sixth section (Khand 6). The chapter, titled “The Story of Bhanu Vinayaka,” establishes him as an aspect connected to the solar principle (Bhanu), the Sun, and the mythic figure of Aruna, the Sun’s charioteer. According to the text, Aruna—having been born from the eggs of Vinata and having worshipped the Sun—encountered Ganesha, bowed before him, and narrated the associated story. Through this interaction, Aruna established the supreme Ganapati upon his chariot and became devoted to the peace of yoga. The passage explicitly states that the greatness of Vikata, pertaining to Bhanu Vinayaka “who is an incarnation of a portion of him,” has been told, and that this account